# Plant Doctor AI — EfficientNet-B0 Baseline

Stage 5 baseline experiment. Dataset A = original PlantVillage training images only; no GAN or traditional augmentation. Validation and test remain deterministic.

In [ ]:
from pathlib import Path
import os, sys, torch

assert torch.cuda.is_available(), 'CUDA GPU is required for Stage 5 training.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

REPO_DIR = Path('/content/Plant-Doctor-AI')
if not REPO_DIR.exists():
    !git clone https://github.com/atharavakadam21-crypto/Plant-Doctor-AI.git /content/Plant-Doctor-AI
%cd /content/Plant-Doctor-AI
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)

In [ ]:
# Dataset was prepared on Windows and stored in Drive as a ZIP.
# Normalize backslashes while extracting because Linux treats '\\' as a filename character.
from pathlib import Path
import shutil, zipfile

ZIP_PATH = Path('/content/drive/MyDrive/Colab Notebooks/plant_doctor_dataset.zip')
DATASET_ROOT = Path('/content/plant_doctor_dataset')

if not all((DATASET_ROOT / s).exists() for s in ['train', 'validation', 'test']):
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'Dataset ZIP not found: {ZIP_PATH}')
    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        for member in z.infolist():
            name = member.filename.replace('\\', '/')
            target = DATASET_ROOT / name
            if member.is_dir() or name.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

print('Dataset root:', DATASET_ROOT)
print('Splits:', [(s, (DATASET_ROOT / s).exists()) for s in ['train', 'validation', 'test']])

In [ ]:
# Verify exact sample counts and all 8 classes before training.
from preprocessing.dataset_loader import CLASS_NAMES

expected_counts = {
    'train': [1394, 2068, 1398, 212, 1400, 2674, 2228, 200],
}
# Counts above are in the current canonical order: Tomato healthy, Tomato EB, Tomato LB,
# Potato healthy, Potato EB, Potato LB, Pepper healthy, Pepper bacterial spot.
expected = {
    'train': [2228, 1400, 2674, 212, 1398, 1400, 2068, 1394],
    'validation': [636, 400, 762, 60, 402, 400, 592, 400],
    'test': [318, 200, 382, 32, 200, 200, 296, 200],
}
grand_total = 0
for split, counts in expected.items():
    observed = []
    for cls in CLASS_NAMES:
        observed.append(len([p for p in (DATASET_ROOT / split / cls).iterdir() if p.is_file()]))
    print(split, observed, 'PASS' if observed == counts else 'FAIL')
    assert observed == counts, f'{split} counts mismatch: {observed} vs {counts}'
    grand_total += sum(observed)
print('Grand total:', grand_total)
assert grand_total == 18254
print('Dataset integrity gate: PASSED')

In [ ]:
# Training configuration. Results/checkpoints are stored in Drive.
DRIVE_RESULTS = Path('/content/drive/MyDrive/Plant-Doctor-AI/results/efficientnet_b0_baseline')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Results directory:', DRIVE_RESULTS)
!python training/efficientnet_b0_train.py --dataset-root {DATASET_ROOT} --output-dir {DRIVE_RESULTS} --epochs 15 --batch-size 32 --num-workers 2 --lr 3e-4 --weight-decay 1e-4 --early-stop-patience 5 --seed 42

In [ ]:
# Inspect measured results only; do not enter or infer metrics manually.
import json
metrics_path = DRIVE_RESULTS / 'metrics.json'
history_path = DRIVE_RESULTS / 'training_history.csv'
print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
print('History:', history_path)
print('Checkpoint:', DRIVE_RESULTS / 'best_efficientnet_b0.pt')